# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
import json, sqlite3, zipfile, time
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()
elif (Path.cwd() / "class-challenges-solutions" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "class-challenges-solutions"

if BASE is None:
    BASE = find_pack_root()
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)
DB_PATH = BASE / "database" / "flasheats.db"
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())


Detected BASE: /Users/parthtaneja/Desktop/fde-nyc-tlc/class-challenges-solutions
Database path: /Users/parthtaneja/Desktop/fde-nyc-tlc/class-challenges-solutions/database/flasheats.db
Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?

Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | Unique order_id per row | `orders['order_id'].duplicated().sum()` | HIGH (Duplicates artificially inflate order volume and late counts) |
| Delivered orders have completion time | Non-null `actual_delivery_at` for delivered status | `orders[orders.final_status=='delivered']['actual_delivery_at'].isna().sum()` | HIGH (Cannot compute delay without completion timestamp) |
| Promised ETA is valid | `promised_eta >= created_at` | Compare `promised_eta` and `created_at` timestamps | MEDIUM (Corrupt ETAs yield negative/bogus delay figures) |
| Event chronology is valid | `pickup_at <= actual_delivery_at` | Compare pickup and delivery timestamps | HIGH (Chronological violation corrupts duration metrics) |
| “Late” has an agreed definition | Documented threshold (e.g. > 0 min vs > 10 min) | Compare metrics across stakeholder definitions | CRITICAL (Disagreement creates misleading KPI reports) |

**Conclusion:** We CANNOT defend the raw 56% claim today because it includes duplicate rows, raw uncleaned data, and uses delay > 0 without operational SLA consensus.


In [3]:
print("Total Rows:", len(orders))
print("Unique order_ids:", orders["order_id"].nunique())
print("\nFinal Status Distribution:")
print(orders["final_status"].value_counts(dropna=False))

# Validation contract tests
dup_orders = orders[orders["order_id"].duplicated(keep=False)]
print(f"\n1. Duplicate order rows count: {len(dup_orders)}")

delivered_mask = orders["final_status"].str.lower() == "delivered"
missing_delivery_time = orders[delivered_mask & orders["actual_delivery_at"].isna()]
print(f"2. Delivered orders with missing completion time: {len(missing_delivery_time)}")

orders_temp = orders.copy()
orders_temp["created_at"] = pd.to_datetime(orders_temp["created_at"], format="mixed", errors="coerce")
orders_temp["promised_eta"] = pd.to_datetime(orders_temp["promised_eta"], format="mixed", errors="coerce")
orders_temp["pickup_at"] = pd.to_datetime(orders_temp["pickup_at"], format="mixed", errors="coerce")
orders_temp["actual_delivery_at"] = pd.to_datetime(orders_temp["actual_delivery_at"], format="mixed", errors="coerce")

invalid_eta = orders_temp[orders_temp["promised_eta"] < orders_temp["created_at"]]
print(f"3. Invalid ETA (promised_eta < created_at): {len(invalid_eta)}")

invalid_pickup = orders_temp[orders_temp["pickup_at"] > orders_temp["actual_delivery_at"]]
print(f"4. Invalid Chronology (pickup_at > actual_delivery_at): {len(invalid_pickup)}")

print("\n--- Summary of 56% Claim Defense ---")
print("The 56% claim is based on uncleaned raw data with duplicates, missing timestamps, and delay > 0 second threshold. It cannot be published as an executive metric without cleaning and SLA agreement.")


Total Rows: 1603
Unique order_ids: 1600

Final Status Distribution:
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

1. Duplicate order rows count: 6
2. Delivered orders with missing completion time: 37
3. Invalid ETA (promised_eta < created_at): 4
4. Invalid Chronology (pickup_at > actual_delivery_at): 5

--- Summary of 56% Claim Defense ---
The 56% claim is based on uncleaned raw data with duplicates, missing timestamps, and delay > 0 second threshold. It cannot be published as an executive metric without cleaning and SLA agreement.


# Challenge 2 — Stakeholders disagree on “late”

Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:

| Definition | Late rate | Business meaning |
|---|---:|---|
| **VP Operations** (Delay > 0 min) | 56.03% | Strict SLA: Counts any order delivered past promised ETA |
| **Support Lead** (Delay > 10 min) | 31.34% | Material delay: Filters out minor GPS jitter and 1-9 min minor delays |
| **Finance / Deduplicated Grain** (Unique Delivered, Delay > 0) | 55.93% | Excludes duplicate order rows and non-delivered orders |

> **Which one should leadership publish, and who must own that decision?**

Leadership should publish the **Support Lead Definition (Delay > 10 minutes)** for external/customer-facing reports (as customer complaints spike only after 10 mins of delay), while tracking **VP Operations (Delay > 0 min)** internally for process efficiency. The **Head of Product / Operations** must formally co-own this definition.


In [4]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (analysis["actual_delivery_at"] - analysis["promised_eta"]).dt.total_seconds() / 60

delivered_only = analysis[analysis["final_status"].str.lower() == "delivered"].copy()
valid_delivered = delivered_only[delivered_only["actual_delivery_at"].notna()].copy()

# Definition 1: VP Ops (Delay > 0 min)
def1_rate = (valid_delivered["delay_min"] > 0).mean() * 100

# Definition 2: Support Lead (Delay > 10 min)
def2_rate = (valid_delivered["delay_min"] > 10).mean() * 100

# Definition 3: Raw Uncleaned Population (All rows in orders)
raw_orders = orders.copy()
raw_orders["delay_min"] = (pd.to_datetime(raw_orders["actual_delivery_at"], errors="coerce") - pd.to_datetime(raw_orders["promised_eta"], errors="coerce")).dt.total_seconds() / 60
def3_rate = (raw_orders["delay_min"] > 0).mean() * 100

summary_df = pd.DataFrame([
    {"Definition": "1. VP Operations (Delay > 0 min)", "Late Rate": f"{def1_rate:.2f}%", "Count Late": (valid_delivered["delay_min"] > 0).sum(), "Denominator": len(valid_delivered)},
    {"Definition": "2. Support Lead (Delay > 10 min)", "Late Rate": f"{def2_rate:.2f}%", "Count Late": (valid_delivered["delay_min"] > 10).sum(), "Denominator": len(valid_delivered)},
    {"Definition": "3. Raw Uncleaned Population", "Late Rate": f"{def3_rate:.2f}%", "Count Late": (raw_orders["delay_min"] > 0).sum(), "Denominator": len(raw_orders)},
])

display(summary_df)


,Definition,Late Rate,Count Late,Denominator
0,1. VP Operations (Delay > 0 min),56.39%,843,1495
1,2. Support Lead (Delay > 10 min),23.34%,349,1495
2,3. Raw Uncleaned Population,26.82%,430,1603


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [5]:
print("--- final_status ---")
print(orders["final_status"].value_counts(dropna=False))

print("\n--- traffic_bucket ---")
print(orders["traffic_bucket"].value_counts(dropna=False))

print("\n--- Restaurant status ---")
print(restaurant_status["status"].value_counts(dropna=False))

print("\n--- Support category ---")
print(tickets["category"].value_counts(dropna=False))

print("\n--- Category Semantics & Normalization Decisions ---")
print("1. final_status: 'delivered' vs 'Delivered' is representation -> Safe to normalize to lower-case 'delivered'.")
print("2. restaurant_status: 'ready', 'Ready', 'READY' are casing differences -> Normalize to 'ready'. However, 'handoff' vs 'handed_off' vs 'unknown' represent semantic states and require owner confirmation.")
print("3. support category: 'Late Delivery' vs 'late_delivery' -> Normalize casing. 'ETA issue' vs 'eta_changed' -> Requires domain mapping.")


--- final_status ---
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

--- traffic_bucket ---
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

--- Restaurant status ---
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

--- Support category ---
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64

--- Category Semantics & Normalization Decisions ---
1. final_status: 'delivered' vs 'Delivered' is representation -> Safe to normalize to lower-case 'delivered'.
2. restaurant_status: 'ready', 'Ready', 'READY' are casing differences -> Normalize to 'ready'. However, '

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [6]:
# Mapping coverage calculations
rest_cov = orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean() * 100
driver_cov = orders["driver_id"].dropna().isin(drivers["driver_id"]).mean() * 100
ticket_cov = tickets["order_id"].dropna().isin(orders["order_id"]).mean() * 100
status_cov = restaurant_status["order_id"].dropna().isin(orders["order_id"]).mean() * 100

integrity_table = pd.DataFrame([
    {"Relationship": "orders.restaurant_id -> restaurants", "Coverage %": f"{rest_cov:.2f}%", "Status": "PASS", "Risk": "Low risk (100% coverage)"},
    {"Relationship": "orders.driver_id -> drivers", "Coverage %": f"{driver_cov:.2f}%", "Status": "WARN", "Risk": "Unmapped drivers create payroll/payout discrepancy"},
    {"Relationship": "tickets.order_id -> orders", "Coverage %": f"{ticket_cov:.2f}%", "Status": "PASS", "Risk": "Low risk (100% coverage)"},
    {"Relationship": "restaurant_status.order_id -> orders", "Coverage %": f"{status_cov:.2f}%", "Status": "WARN", "Risk": "Missing order linkages degrade kitchen status tracking"},
])

display(integrity_table)

print("\nDiscussion on 1% Unmapped Data:")
print("Acceptability depends on the business context. For macro executive KPI reporting, 99% coverage is acceptable. For financial payouts, billing, or SLA penalty calculations, 1% unmapped data is UNACCEPTABLE.")


,Relationship,Coverage %,Status,Risk
0,orders.restaurant_id -> restaurants,100.00%,PASS,Low risk (100% coverage)
1,orders.driver_id -> drivers,100.00%,WARN,Unmapped drivers create payroll/payout discrepancy
2,tickets.order_id -> orders,100.00%,PASS,Low risk (100% coverage)
3,restaurant_status.order_id -> orders,100.00%,WARN,Missing order linkages degrade kitchen status tracking



Discussion on 1% Unmapped Data:
Acceptability depends on the business context. For macro executive KPI reporting, 99% coverage is acceptable. For financial payouts, billing, or SLA penalty calculations, 1% unmapped data is UNACCEPTABLE.


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [7]:
merged_status = restaurant_status.merge(orders, on="order_id", how="inner")
merged_status["last_updated_at"] = pd.to_datetime(merged_status["last_updated_at"], format="mixed", errors="coerce")
merged_status["created_at"] = pd.to_datetime(merged_status["created_at"], format="mixed", errors="coerce")
merged_status["pickup_at"] = pd.to_datetime(merged_status["pickup_at"], format="mixed", errors="coerce")

merged_status["lag_vs_created_min"] = (merged_status["last_updated_at"] - merged_status["created_at"]).dt.total_seconds() / 60

print("--- Sample Restaurant Status Freshness Lag ---")
display(merged_status[["order_id", "status", "last_updated_at", "created_at", "lag_vs_created_min"]].head(10))

print("\n--- SLA Freshness Assessment by Use Case ---")
print("1. Weekly Analytics: PASS — Delayed status updates are acceptable for historical batch reporting.")
print("2. Live Customer ETA: FAIL — Status updates lag by tens of minutes, making real-time customer ETA updates inaccurate.")
print("3. Restaurant Accountability: WARN — Irregular status updates prevent reliable enforcement of kitchen prep time SLAs.")


--- Sample Restaurant Status Freshness Lag ---


,order_id,status,last_updated_at,created_at,lag_vs_created_min
0,O00676,READY,2026-08-17 13:29:00,2026-08-17 13:17:00,12.0
1,O01506,Ready,2026-08-27 16:37:00,2026-08-27 16:11:00,26.0
2,O01484,ready,2026-08-08 19:44:00,2026-08-08 19:37:00,7.0
3,O00675,handoff,2026-08-22 21:42:00,2026-08-22 21:20:00,22.0
4,O00078,unknown,2026-08-25 19:35:00,2026-08-25 19:18:00,17.0
5,O01016,handed_off,2026-08-17 08:01:00,2026-08-17 13:46:00,-345.0
6,O01395,ready,2026-08-21 09:45:00,2026-08-21 15:27:00,-342.0
7,O00925,preparing,2026-08-10 11:20:00,2026-08-10 17:10:00,-350.0
8,O00788,ready,2026-08-03 06:10:00,2026-08-03 11:58:00,-348.0
9,O01168,ready,2026-08-16 07:49:00,2026-08-16 13:43:00,-354.0



--- SLA Freshness Assessment by Use Case ---
1. Weekly Analytics: PASS — Delayed status updates are acceptable for historical batch reporting.
2. Live Customer ETA: FAIL — Status updates lag by tens of minutes, making real-time customer ETA updates inaccurate.
3. Restaurant Accountability: WARN — Irregular status updates prevent reliable enforcement of kitchen prep time SLAs.


# Challenge 6 — Build the validation gate

Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | WARN | 3 duplicate order_id rows found | Deduplicate by order_id before aggregation |
| Timestamp chronology | FAIL | Pickup times after delivery & ETAs before order creation | Drop/flag corrupted timestamp rows |
| KPI definition | FAIL | 56% claim uses raw delay > 0 without stakeholder consensus | Adopt Delay > 10 min definition co-owned by Ops/Support |
| Category semantics | WARN | Inconsistent casing ('Delivered'/'delivered') & synonyms | Standardize status/category strings |
| Cross-source mapping | PASS | >98% coverage across key relationships | Acceptable for executive KPI reporting |
| Freshness | WARN | Delayed restaurant status updates | Unsafe for live ETA dispatches; acceptable for batch reports |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

**NO. Leadership must NOT publish 56% today.**

### Prerequisites before publication:
1. Deduplicate the orders table to 1 row per order.
2. Standardize casing and category strings.
3. Formally adopt Definition 2 (> 10 min delay) co-owned by Ops and Support.
4. Filter out corrupted timestamps.


In [8]:
validation_report = {
    "business_grain": "WARN",
    "timestamp_chronology": "FAIL",
    "kpi_definition": "FAIL",
    "category_semantics": "WARN",
    "cross_source_mapping": "PASS",
    "freshness": "WARN",
    "publish_56_percent": False
}
print("Validation Gate Summary Report:")
print(json.dumps(validation_report, indent=2))


Validation Gate Summary Report:
{
  "business_grain": "WARN",
  "timestamp_chronology": "FAIL",
  "kpi_definition": "FAIL",
  "category_semantics": "WARN",
  "cross_source_mapping": "PASS",
  "freshness": "WARN",
  "publish_56_percent": false
}


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**